<div style="color: green; font-weight: bold">Part 1 Comment</div>

This solutions utilizes 0.5 as b_MOR and a different activation function than the sample but results in the same correct result of the operations given.

The solution uses 4 lines to draw boundaries instead of the sample's 3 lines creating six boundary regions. The sample solution creates a one-hot equation that contains 7 regions to account for a region below the plot. This solution does not account for that 7th region and instead makes one-hot equations in the second layer that only contain 6 regions. The drawn neurons for layer 2 end up being only 5 when I think they should be 6 (or if accounting for boundary below the plot like the sample 7 neurons). The solution uses a perfect match for layer 2 and a masked OR which is essentially equal to the sample solution.

The solution could say more about how the network could be generalized to arbitrary many input dimension and arbitrary label distributions. The identified problems are essentially the same as the sample solution and have good explanations on why they are problems.

# Exercise 2

## Part 3 — commented feedback version

In [1]:
import numpy as np
from sklearn import datasets

In [2]:
class ReLULayer(object):
    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # ReLU: element wise max with 0
        relu = np.maximum(0, input)
        return relu

    def backward(self, upstream_gradient):
        # derivative of ReLU is 1 where input > 0, else 0
        # by chain rule: dL/dZ_{l-1} = dL/dZ_l * ReLU'(input)
        downstream_gradient = upstream_gradient * (self.input > 0)
        return downstream_gradient

    def update(self, learning_rate):
        pass  # ReLU is parameter free


<div style="color: green; font-weight: bold">Comment</div>

**ReLULayer:**
This implementation is correct and compact. The forward pass uses the vectorized expression `np.maximum(0, input)`, which is equivalent to the sample solution. The backward pass multiplies the upstream gradient by the ReLU derivative mask `(self.input > 0)`, which is also correct and avoids slow Python loops.

The only small difference from the sample solution is how the derivative at exactly zero is handled. Since ReLU is not differentiable at zero, choosing derivative `0` there is standard and consistent with the sample solution's behavior. The layer is parameter-free, so `update` correctly does nothing.

In [3]:
class OutputLayer(object):
    def __init__(self, n_classes):
        self.n_classes = n_classes

    def forward(self, input):
        # remember the input (= pre softmax logits) for backprop
        self.input = input
        # numerically stable softmax: subtract row wise max before exp
        shifted = input - np.max(input, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        softmax = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        return softmax

    def backward(self, predicted_posteriors, true_labels):
        # combined gradient of softmax + cross entropy (as derived in lecture):
        # dL/dz = (p - y_onehot) / batch_size
        # this elegant form is the reason softmax + CE are paired
        batch_size = predicted_posteriors.shape[0]
        one_hot = np.zeros_like(predicted_posteriors)
        one_hot[np.arange(batch_size), true_labels] = 1
        downstream_gradient = (predicted_posteriors - one_hot) / batch_size
        return downstream_gradient

    def update(self, learning_rate):
        pass  # softmax is parameter free


<div style="color: green; font-weight: bold">Comment</div>

**OutputLayer:**
The softmax forward pass is correct and numerically stable because the row-wise maximum is subtracted before applying `exp`. This is a slightly more standard version of the stabilization used in the sample solution.

The backward pass is also correct: it explicitly constructs the one-hot encoding of the true labels and returns `(predicted_posteriors - one_hot) / batch_size`. This is mathematically equivalent to the sample solution, where `1` is subtracted from the posterior of the correct class in place. The implementation is fully vectorized and readable.

In [4]:
class LinearLayer(object):
    def __init__(self, n_inputs, n_outputs):
        self.n_inputs = n_inputs
        self.n_outputs = n_outputs
        # He initialization: std = sqrt(2 / fan_in)
        # well suited for ReLU networks (keeps signal variance ~constant)
        self.B = np.random.normal(0, np.sqrt(2.0 / n_inputs), (n_inputs, n_outputs))
        # biases initialized to zero (a small constant also works)
        self.b = np.zeros(n_outputs)

    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # pre activations: Z_l = Z_{l-1} @ B + b
        preactivations = input @ self.B + self.b
        return preactivations

    def backward(self, upstream_gradient):
        # parameter gradients (chain rule):
        #   dL/dB = input.T @ upstream_gradient   (sums contributions across batch)
        #   dL/db = sum of upstream_gradient over the batch dimension
        self.grad_B = self.input.T @ upstream_gradient
        self.grad_b = upstream_gradient.sum(axis=0)
        # downstream gradient (to be passed to preceding layer):
        #   dL/dZ_{l-1} = upstream_gradient @ B.T
        downstream_gradient = upstream_gradient @ self.B.T
        return downstream_gradient

    def update(self, learning_rate):
        # batch gradient descent step
        self.B = self.B - learning_rate * self.grad_B
        self.b = self.b - learning_rate * self.grad_b

<div style="color: green; font-weight: bold">Comment</div>

**LinearLayer:**
The layer is implemented correctly. The forward pass computes the affine transformation `input @ B + b`, and the backward pass computes all required gradients with vectorized matrix operations: `input.T @ upstream_gradient` for the weights, a batch-wise sum for the bias, and `upstream_gradient @ B.T` for the downstream gradient.

Compared with the sample solution, the initialization differs: this version uses He initialization with standard deviation `sqrt(2 / n_inputs)` and zero biases. This is a valid alternative and is even more appropriate for ReLU networks, because it helps keep activation variances stable across layers. The update step is standard batch gradient descent and is correct.

In [5]:
class MLP(object):
    def __init__(self, n_features, layer_sizes):
        # construct a multi layer perceptron
        # with ReLU activation in the hidden layers and softmax output
        self.n_layers = len(layer_sizes)
        self.layers = []

        # interior layers: Linear + ReLU
        n_in = n_features
        for n_out in layer_sizes[:-1]:
            self.layers.append(LinearLayer(n_in, n_out))
            self.layers.append(ReLULayer())
            n_in = n_out

        # final linear layer + softmax output
        n_out = layer_sizes[-1]
        self.layers.append(LinearLayer(n_in, n_out))
        self.layers.append(OutputLayer(n_out))

    def forward(self, X):
        # X is a mini batch of instances
        batch_size = X.shape[0]
        # flatten in case instances are images
        X = X.reshape(batch_size, -1)
        # forward pass: each layer also stores its input for backprop
        result = X
        for layer in self.layers:
            result = layer.forward(result)
        return result

    def backward(self, predicted_posteriors, true_classes):
        # backpropagation: walk the layers in reverse
        # the output layer needs both posteriors and true labels;
        # every other layer just receives the upstream gradient
        gradient = self.layers[-1].backward(predicted_posteriors, true_classes)
        for layer in reversed(self.layers[:-1]):
            gradient = layer.backward(gradient)

    def update(self, X, Y, learning_rate):
        posteriors = self.forward(X)
        self.backward(posteriors, Y)
        for layer in self.layers:
            layer.update(learning_rate)

    def train(self, x, y, n_epochs, batch_size, learning_rate):
        N = len(x)
        n_batches = N // batch_size
        for i in range(n_epochs):
            # shuffle data each epoch (mini batches without replacement)
            permutation = np.random.permutation(N)
            for batch in range(n_batches):
                start = batch * batch_size
                x_batch = x[permutation[start:start + batch_size]]
                y_batch = y[permutation[start:start + batch_size]]
                self.update(x_batch, y_batch, learning_rate)

<div style="color: green; font-weight: bold">Comment</div>

**MLP class:**
The backpropagation is correct and equivalent to the sample solution. Using `reversed(self.layers[:-1])` is a clear and readable way to iterate backwards through all layers except the output layer. A minor naming point is that `self.n_layers` stores the number of entries in `layer_sizes`, not the actual number of layer objects in `self.layers`, but this does not affect the computation.

In [ ]:
if __name__ == "__main__":
    # set training / test set size
    N = 2000

    # create training and test data
    X_train, Y_train = datasets.make_moons(N, noise=0.05)
    X_test,  Y_test  = datasets.make_moons(N, noise=0.05)

    n_features = 2
    n_classes = 2

    # standardize features to be in [-1, 1]
    offset = X_train.min(axis=0)
    scaling = X_train.max(axis=0) - offset
    X_train = ((X_train - offset) / scaling - 0.5) * 2.0
    X_test  = ((X_test  - offset) / scaling - 0.5) * 2.0

    # Fixed training hyperparameters
    n_epochs = 50
    batch_size = 200
    learning_rate = 0.1

    # 4 networks
    network = [
        [2, 2, n_classes],
        [3, 3, n_classes],
        [5, 5, n_classes],
        [30, 30, n_classes]
    ]

    # Loop through each network, train it, and print its validation error
    for sizes in architectures:
        # np.random.seed(42)  # Optional: uncomment if you want reproducible results across runs
        
        network = MLP(n_features, sizes)
        network.train(X_train, Y_train, n_epochs, batch_size, learning_rate)
        
        predicted_posteriors = network.forward(X_test)
        predicted_classes = np.argmax(predicted_posteriors, axis=1)
        error_rate = np.mean(predicted_classes != Y_test)
        
        print(f"Network {sizes[:-1]} -> Test Error Rate: {error_rate:}")

Network [2, 2] -> Test Error Rate: 0.105
Network [3, 3] -> Test Error Rate: 0.108
Network [5, 5] -> Test Error Rate: 0.0295
Network [30, 30] -> Test Error Rate: 0.0045


<div style="color: green; font-weight: bold">Comment</div>

**Training and testing block:**
The overall idea is good and extends the sample solution by comparing several network architectures. The preprocessing, training call, posterior prediction, `argmax` class decision, and error-rate computation are all conceptually correct.

However, there is a bug: the architecture list is assigned to the variable `network`, but the loop uses `for sizes in architectures:`. Since `architectures` is never defined, this code will raise a `NameError`. A brief fix would be to store the list under the name `architectures`, or to loop over the existing list without reusing the name `network` for both the list and the trained model.


Network [2, 2] (Error Rate: 11.40%): Two hidden units can only create a single corner bend or wedge split using ReLU. It separates the main clusters but lacks the capacity to wrap around the winding crescent tips, causing a high error rate.

Network [3, 3] (Error Rate: 9.90%): The third neuron adds a dimension to the hidden space, letting the network combine three hyperplanes to form a slightly tighter polygon around the data curves.

Network [5, 5] (Error Rate: 4.90%): Five units allow the network to stitch together enough linear segments to cleanly approximate smooth, winding decision boundaries through the gap between the moons.

Network [30, 30] (Error Rate: 0.10%): The large width provides massive capacity and creates a benign optimization landscape with many redundant gradient paths, allowing the model to converge almost instantly within 5 epochs.